# Regression and classification template

This follows https://auto.gluon.ai/stable/tutorials/tabular/tabular-quick-start.html

- Work through the notebook cell by cell
- Make changes to make it work for your project


### Imports

In [1]:
import pandas as pd
import numpy as np
from psmiles import PolymerSmiles as PS
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor

### Data


In [ ]:
df_init = pd.read_json("data.json")
df_init

### Scale the fingerprints and targets

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Scale the fingerprints
scaler_fps = MinMaxScaler()
# fps is an arry with shape (n_samples, n_features) containing the fingerprints
fps_scaled = pd.DataFrame(scaler_fps.fit_transform(fps))

# Scale the target(s)
scaler_targets = MinMaxScaler()
# write back to new column
df["target_scaled"] = scaler_targets.fit_transform(df[["target"]])

# Save the scalers to disk for later use
import joblib
joblib.dump(scaler_fps, "scaler_fps.pkl")
joblib.dump(scaler_targets, "scaler_targets.pkl")

### Prepare final data frame

In [ ]:
# Concat fingerprints
df = pd.concat((target_scaled, fps_scaled), axis=1)

# Make sure they're all float
df = df.astype(np.float32)

# Remove columns that are zero, if any
df = df.loc[:, (df != 0).any(axis=0)]


### Split in train and test 

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.20, random_state=42)

display(df_train)
display(df_test)

### Train you AutoGluon ML model

In [ ]:
predictor = TabularPredictor(
    label="value",
    problem_type="regression", # or classification, depending on the problem
).fit(df_train, time_limit=60, presets="best_quality")

### Use matplotlib for plotting

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error

y_pred = predictor.predict(df_test.drop(columns=["value"]))

r2 = r2_score(df_test["value"], y_pred)
rmse = root_mean_squared_error(df_test["value"], y_pred)

print(f"R2: {r2}")
print(f"RMSE: {rmse}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

y_pred = predictor.predict(df_test.drop(columns=["value"]))
y_pred

ax.plot(y_pred, df_test["value"], "o")
ax.plot([0, 1], [0, 1], "k--")
ax.text(0.1, 0.9, f"R2 = {r2:.3f}", transform=ax.transAxes)
ax.text(0.1, 0.85, f"RMSE = {rmse:.3f}", transform=ax.transAxes)
ax.set_title("Testdata set")
ax.set_ylabel("truth")
ax.set_xlabel("pred")